In [ ]:
import pandas as pd
import sqlite3
import paramiko
import tempfile
import os


def read_db_directly():
    """Read the database and return as dataframe"""
    # Write key content to temporary file
    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.key') as temp_key_file:
        temp_key_file.write(ssh_private_key)
        temp_key_path = temp_key_file.name
    
    config = {
        'host': '80.225.228.224',
        'username': 'ubuntu',
        'private_key': temp_key_path,
    }
    
    remote_db_path = '/home/ubuntu/final_trading_logs.db'
    
    try:
        private_key = paramiko.RSAKey.from_private_key_file(config['private_key'])
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(
            hostname=config['host'],
            username=config['username'],
            pkey=private_key
        )
        
        with tempfile.NamedTemporaryFile(suffix='.db', delete=False) as temp_file:
            temp_path = temp_file.name
        
        sftp = ssh.open_sftp()
        sftp.get(remote_db_path, temp_path)
        sftp.close()
        
        conn = sqlite3.connect(temp_path)
        df = pd.read_sql_query("SELECT * FROM trading_logs", conn)
        conn.close()
        
        os.unlink(temp_path)
        ssh.close()
        
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None

# Read the data
df = read_db_directly()